# 04 – Backtesting Analysis

**Goal:** Evaluate the parametric VaR model using historical daily data.
Count VaR exceedances and apply the Kupiec Proportion of Failures (PoF) test.

**Prerequisite:** Notebooks 01 and 02 must have been run.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import config
from src.data_loader  import load_saved_data
from src.preprocess   import compute_log_returns
from src.portfolio    import initialize_portfolio
from src.covariance   import rolling_covariance
from src.backtesting  import summarize_backtest_results

## 1. Load Data

In [ ]:
daily       = load_saved_data('daily_prices.csv', config.DATA_RAW)
log_returns = compute_log_returns(daily)

# Use the first available price row to initialise the portfolio
portfolio = initialize_portfolio(daily.iloc[0])
shares    = portfolio['shares']
cov_roll  = rolling_covariance(log_returns)

## 2. Run Backtesting

In [ ]:
backtest = summarize_backtest_results(
    daily_prices = daily,
    shares       = shares,
    log_returns  = log_returns,
    cov_matrix   = cov_roll,
    save         = True,
)
backtest

## 3. Exceedance Plot

In [ ]:
# Load the detailed breach series produced by backtesting.py
breach_series = load_saved_data('backtest_series.csv', config.DATA_PROCESSED)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(breach_series['actual_loss'],  label='Actual Loss',  color='navy')
ax.plot(breach_series['var_99'],       label='99% VaR',      color='crimson', linestyle='--')

breaches = breach_series[breach_series['breach_99']]
ax.scatter(breaches.index, breaches['actual_loss'],
           color='red', zorder=5, label=f'Breach ({len(breaches)} events)')

ax.set_title('Backtesting: Actual Loss vs 99% VaR Threshold')
ax.set_ylabel('Dollar Loss ($)')
ax.legend()
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/backtesting_exceedances.png', dpi=150)
plt.show()